# modular-diffusion: exchanging named parts of the toy collective

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/AbstractEyes/modular-diffusion/blob/v0.16.0/colab/toy_swap.ipynb)

Sampling only, nothing trained. THE TOY COLLECTIVE (`colab/toy_collective.ipynb`): nine small engines that each learned to draw one thing in one cell of a 3 x 3 grid, alone, and a base engine for the cells without a thing; every image token of a picture is served by the one engine whose declared address it carries, and a small judge reads each cell. Its parts were trained for 3,000 steps. Some parts were since trained again, alone, for longer (6,000 and 12,000 steps), or wider (width 192, 3 heads).

THIS NOTEBOOK serves a finished collective with only some of its parts EXCHANGED for those other finished parts (the library's `swap` argument: `class:engine folder`, or `newborn` for an untrained part), everything else untouched. Every job draws the same request lists (or captions) from the same start noise twice, with the exchanged collective and with the unexchanged one, and stores the largest pixel difference outside the cells of the exchanged things (it must be exactly zero) beside both collectives' reads, so that the two pair list by list and caption by caption. THE ROWS (the table at the top of the second cell): the star exchanged for its 6,000-step part; the disc, the plus, the cross and the star exchanged for theirs; the star for its 12,000-step part; the star for its wider part; all nine parts for their 6,000-step parts; the star for an untrained part (which must fail to draw stars). For every row and seed: the earlier run's 500 request lists of 1, 2, 3, 4, 6 and 9 things, and the 600 captions of the prompt-to-picture notebook drawn from their true layouts and from a language model's stored layouts (`colab/toy_prompt_to_picture.ipynb`; no language model runs here).

1. install at the release tag and print the environment;
2. the earlier runs' jobs fetched from the Hub (the cell stops if one is missing); the job list; a smoke of one exchange on 16 short lists and of one caption job; the projected minutes; the smoke shipped;
3. the job list, one job at a time; each finished job folder shipped at once; a progress line a minute; the summary; the machine is given back only when every finished job and the report were shipped.

RESUME: run the cells again. Finished jobs keep their markers under `OUT`; on a new machine finished jobs are fetched back from the Hub, so nothing runs twice.

In [ ]:
# C1 - install at the tag (extras on the name); torch is not a dependency, so the runtime's torch stays
%pip install -q "modular-diffusion[hub] @ git+https://github.com/AbstractEyes/modular-diffusion@v0.16.0"
import os, sys, time, json, shutil, threading
import torch, geolip.modular_diffusion as MD
assert tuple(int(x) for x in MD.__version__.split(".")) >= (0, 16, 0), "RESTART REQUIRED: Runtime > Restart session, then run C1 again"
import huggingface_hub
print("torch", torch.__version__, "| huggingface_hub", huggingface_hub.__version__, "| package", MD.__version__)
if torch.cuda.is_available():
    props = torch.cuda.get_device_properties(0)
    print("card:", props.name, f"| memory {props.total_memory / 2**30:.1f} GiB | cpus: {os.cpu_count()}")
else:
    print("card: NONE (choose a GPU runtime)")
OUT = "/content/toy_swap"       # the output root; keep it the same to resume
os.makedirs(OUT, exist_ok=True)

The earlier runs' jobs are fetched from the Hub: the judge, the shipped 3,000-step parts and bases of both seeds, the stored rows of the collective, the longer and the wider parts, and the two stored writer cells (the true layouts and a language model's answers on the 600 captions). The judges of the `JUDGES` list (the earlier run's judge first, then four more of the same recipe with other seeds) are fetched too and every job's pictures are read by each of them; the first judge's reads fill every key a single judge would. The cell stops if a job or a judge is missing, so nothing is ever trained here. The job list is built from the `ROWS` table (a row is run for a seed only when every part it names is found in the fetched jobs; a row can be added to the table without touching the code below it). Then a smoke: one row's exchange on 16 lists of two things, and its picture job of the true layouts on all 600 captions, with their seconds; the job list's minutes are projected from them; the smoke folder is shipped.

In [ ]:
# C2 - the earlier runs, the job list, the smoke, the projection
from huggingface_hub import snapshot_download
from geolip.modular_diffusion import hub, run
from geolip.modular_diffusion import collective as C
from geolip.modular_diffusion.experiments import toy_collective as T
RUNS, SMOKE, FETCHED = f"{OUT}/runs", f"{OUT}/smoke", f"{OUT}/earlier"
REPO = "AbstractPhil/geolip-multirotary-diffusion-collective"   # the results repository ("" = nothing shipped)
PREFIX = "modular_diffusion/conditioning/toy_swap"
EARLIER = "modular_diffusion/conditioning/toy_collective"
WRITTEN = "modular_diffusion/conditioning/toy_prompt_to_picture"
NOT_SHIPPED = ("SHIPPED", "FROM_HUB")
DEVICE = "cuda"
EXTRA = ""                          # more arguments for every job (empty in a real run)
PART_STEPS = 3000                   # the steps of the shipped collective's parts
SHAPE = (128, 2, 3)                 # the shipped parts' shape: width, heads, blocks
LONG, LONGER = 6000, 12000          # the longer parts' steps
WIDE = (192, 3, 3)                  # the wider part's shape
LISTS = "ns=1,2,3,4,6,9 count=500"  # the earlier run's request lists
SMOKE_LISTS = "ns=2 count=16"
# THE JUDGES: every picture is read by each (the first fills every key a single judge would); (job, where it is on the Hub)
JUDGES = [("judge_F", EARLIER), ("judge_F_j1", PREFIX), ("judge_F_j2", PREFIX), ("judge_F_j3", PREFIX), ("judge_F_j4", PREFIX)]
# THE JOB LIST. A row exchanges the classes named for the parts of the steps and shape named, found in the fetched job
# named for each seed ("newborn": an untrained part of the job's seed and shape in their place). Every row and seed:
# one compose job on the lists, two draw jobs (the true layouts, the language model's stored layouts).
#   row            seeds    classes          steps      shape  the fetched job holding the parts, a seed
ROWS = [
    ("SWAP1_6000",  (0, 1), (8,),            LONG,      SHAPE, {0: "dose6000_at_F_s0", 1: "e18_p6000_weak_F_s1"}),
    ("SWAP4_6000",  (0, 1), (0, 6, 7, 8),    LONG,      SHAPE, {0: "dose6000_at_F_s0", 1: "e18_p6000_weak_F_s1"}),
    ("SWAP1_12000", (0, 1), (8,),            LONGER,    SHAPE, {0: "e18_p12000_star_F_s0", 1: "e18_p12000_star_F_s1"}),
    ("SWAP1_W192",  (0,),   (8,),            LONG,      WIDE,  {0: "e18_p6000_starw192_F_s0"}),
    ("ALL9_6000",   (0,),   tuple(range(9)), LONG,      SHAPE, {0: "dose6000_at_F_s0"}),
    ("NEWBORN",     (0,),   (8,),            "newborn", SHAPE, {0: None}),
]
WRITERS = (("true", "true_cell_ex1", "true"), ("lucia9b", "lucia9b_cell_ex1", "chosen"))   # (tag word, writer cell, role)
os.makedirs(RUNS, exist_ok=True)
mark = lambda t, m: os.path.exists(f"{RUNS}/{t}/{m}")
E = lambda t: f"{FETCHED}/{EARLIER}/{t}"
W = lambda t: f"{FETCHED}/{WRITTEN}/{t}"

# the earlier runs' jobs; the cell stops if one is missing
EARLIER_JOBS = ("judge_F", "at_F_s0", "ab_F_s0", "at_F_s1", "ab_F_s1", "dose6000_at_F_s0", "R1_F_s0", "R1_F_s1",
                "e18_p6000_weak_F_s1", "e18_p12000_star_F_s0", "e18_p12000_star_F_s1", "e18_p6000_starw192_F_s0")
PART_JOBS = [t for t in EARLIER_JOBS if t not in ("judge_F", "R1_F_s0", "R1_F_s1")]
WRITER_CELLS = tuple(c for _, c, _ in WRITERS)
def earlier_missing():
    # a job is missing when its result is not here, or (the judge, the engines, a writer cell) its files are not
    miss = [t for t in EARLIER_JOBS if not os.path.exists(f"{E(t)}/result.json")]
    miss += [f"the judge {t}" for t, where in JUDGES
             if not (os.path.exists(f"{FETCHED}/{where}/{t}/result.json") and os.path.exists(f"{FETCHED}/{where}/{t}/judge.safetensors"))]
    for t in PART_JOBS:
        found = T.index_engines(E(t))
        want = 9 if t.startswith(("at_", "dose6000_at_")) else 1
        if len(found) < want or not all(os.path.exists(f"{p}/{f}") for p in found.values() for f in C.ENGINE_FILES):
            miss.append(f"the engines of {t}")
    miss += [f"the writer cell {c}" for c in WRITER_CELLS
             if not (os.path.exists(f"{W(c)}/result.json") and os.path.exists(f"{W(c)}/outputs.jsonl"))]
    return sorted(set(miss))
if earlier_missing():
    snapshot_download(repo_id=REPO, allow_patterns=[f"{EARLIER}/{t}/*" for t in EARLIER_JOBS]
                      + [f"{WRITTEN}/{c}/*" for c in WRITER_CELLS] + [f"{where}/{t}/*" for t, where in JUDGES],
                      local_dir=FETCHED)
MISSING = earlier_missing()
assert not MISSING, f"the earlier runs' jobs are not all on the Hub ({', '.join(MISSING)}): STOP, nothing is trained here"
print("the earlier runs' jobs fetched:", ", ".join(EARLIER_JOBS + WRITER_CELLS), "| the judges:", ", ".join(t for t, _ in JUDGES))

# the job list: (tag, arguments, the row, the seed)
def swap_of(seed, classes, steps, shape, where):
    # -> (the swap argument, the folders holding its parts), or None when a part is not found
    if steps == "newborn":
        return ",".join(f"{k}:newborn" for k in classes), []
    folder = E(where[seed])
    found = T.index_engines(folder)
    names = {k: T.engine_tag("addressed_thing", k, "F", seed, steps, *shape) for k in classes}
    if not all(n in found for n in names.values()):
        return None
    return ",".join(f"{k}:{names[k]}" for k in classes), [folder]
def common(seed, swap, folders):
    parts = ",".join([E(f"at_F_s{seed}"), E(f"ab_F_s{seed}")] + folders)
    judges = ",".join(f"{FETCHED}/{where}/{t}" for t, where in JUDGES)
    return (f"parts={parts} judge={judges} bed=F seed={seed} swap={swap} part_steps={PART_STEPS} dim={SHAPE[0]} "
            f"heads={SHAPE[1]} depth={SHAPE[2]} device={DEVICE} det=on")
JOBS, SKIPPED = [], []
for row, seeds, classes, steps, shape, where in ROWS:
    for s in seeds:
        got = swap_of(s, classes, steps, shape, where)
        if got is None:
            SKIPPED.append(f"{row} seed {s}")
            continue
        args = common(s, *got)
        JOBS.append((f"{row}_F_s{s}", f"toy_collective variant=compose row=R1 {LISTS} {args}{EXTRA}", row, s))
        for word, cell, role in WRITERS:
            JOBS.append((f"draw_{word}_{row}_s{s}", f"toy_collective variant=draw answers={W(cell)} role={role} {args}{EXTRA}",
                         row, s))
BY_TAG = {t: (a, r, s) for t, a, r, s in JOBS}
assert len(BY_TAG) == len(JOBS), "two jobs of one tag"
print(f"the job list: {len(JOBS)} jobs of {len(ROWS)} rows" + (f"; not run, their parts are not here: {', '.join(SKIPPED)}" if SKIPPED else ""))
if REPO:
    from google.colab import userdata
    os.environ["HF_TOKEN"] = userdata.get("HF_TOKEN")          # placed in this process's environment only

def fetch_back(tags):
    # finished jobs come back from the Hub with their files, so nothing runs twice
    if not REPO:
        return
    try:
        got = snapshot_download(repo_id=REPO, allow_patterns=[f"{PREFIX}/{t}/*" for t in tags], local_dir=f"{OUT}/hub_back")
    except Exception as e:
        print("nothing fetched back from the Hub:", repr(e)[:200])
        return
    for t in tags:
        src, dst = f"{got}/{PREFIX}/{t}", f"{RUNS}/{t}"
        if mark(t, "DONE") or not os.path.exists(f"{src}/result.json"):
            continue
        shutil.copytree(src, dst, dirs_exist_ok=True)
        open(f"{dst}/DONE", "w").write("fetched back from the Hub\n")
        open(f"{dst}/FROM_HUB", "w").write("fetched back from the Hub\n")

def ship_job(tag, folder):
    if REPO and hub.ship(folder, REPO, f"{PREFIX}/{tag}", f"toy swap: {tag}", ignore=NOT_SHIPPED):
        open(f"{folder}/SHIPPED", "w").write("shipped to the Hub\n")
        return True
    return False

# the smoke: the first row's exchange of seed 0 on 16 lists of two things, and its picture job of the true layouts
assert JOBS, "no row of the table has its parts here: STOP"
shutil.rmtree(SMOKE, ignore_errors=True)
first = JOBS[0][0]                  # the first row's compose job (each row's compose job comes before its draw jobs)
smoke_args = BY_TAG[first][0].replace(LISTS, SMOKE_LISTS)
first_swap = [x for x in smoke_args.split() if x.startswith("swap=")][0][len("swap="):]
smoke_args = smoke_args.replace(f"swap={first_swap}", f"swap={first_swap.split(',')[0]}")      # one pair
t0 = time.time()
sc = run.main("toy_collective", smoke_args.split()[1:] + [f"out={SMOKE}/compose"])
compose_s = time.time() - t0
draw_tag = f"draw_{WRITERS[0][0]}_{BY_TAG[first][1]}_s{BY_TAG[first][2]}"
t0 = time.time()
sd = run.main("toy_collective", BY_TAG[draw_tag][0].split()[1:] + [f"out={SMOKE}/draw"])
draw_s = time.time() - t0
for name, r in (("compose", sc), ("draw", sd)):
    x = r["exact"]
    print(f"smoke {name}: swap {r['swap']}; exact {x['holds']} (largest difference outside {x['largest_outside']}, before "
          f"the clamp {x['largest_outside_unclamped']}, {x['cells_compared']} cells); the exchange changed something "
          f"{x['real']} (exchanged cells that differ {x['inside_share']}); {r['seconds']:.0f} s", flush=True)
    assert x["holds"] is True, f"the smoke's {name} exchange is not exact: STOP"
    assert x["real"] is True, f"the smoke's {name} exchange changed nothing: STOP"

# the projection: a compose job from the smoke's seconds a (list, request or base) forward; a draw job from the smoke's
def units(lists):
    kv = dict(x.split("=") for x in lists.split())
    return int(kv["count"]) * sum(int(n) + 1 for n in kv["ns"].split(","))
COST = {}
for t, a, r, s in JOBS:
    COST[t] = compose_s * units(LISTS) / units(SMOKE_LISTS) if " variant=compose " in f" {a} " else draw_s
INSTRUMENTS = {"rows": [list(map(str, x)) for x in ROWS], "jobs": [{"tag": t, "arguments": a} for t, a, _, _ in JOBS],
               "skipped": SKIPPED, "smoke_seconds": {"compose": compose_s, "draw": draw_s}, "projected_seconds": COST,
               "earlier_jobs": list(EARLIER_JOBS), "writer_cells": list(WRITER_CELLS),
               "card": torch.cuda.get_device_name(0) if torch.cuda.is_available() else "cpu",
               "versions": {"torch": torch.__version__, "package": MD.__version__}}
json.dump(INSTRUMENTS, open(f"{SMOKE}/instruments.json", "w", encoding="utf-8"), indent=1)
left = [t for t, *_ in JOBS if not mark(t, "DONE")]
print(f"the job list: {len(JOBS)} jobs, {len(JOBS) - len(left)} done; about {sum(COST[t] for t in left) / 60:.0f} projected "
      f"minutes for the rest on this card with one runner")
if REPO:
    assert hub.ship(SMOKE, REPO, f"{PREFIX}/_smoke", "toy swap: smoke run", ignore=NOT_SHIPPED), (
        "the smoke folder was not shipped: check the token and the repository before the long run")

One runner works the job list in order. Nothing is trained: if a job of the earlier runs is missing here, the cell stops before the roster starts. Each finished job folder is shipped at once to `REPO` (a compose job with both collectives' reads by the number of things and its exactness read; a picture job with every caption's record, both collectives' tables and its sheet), with the token from the notebook's secret `HF_TOKEN` (placed in this process's environment only; the jobs themselves run without it). At the end the summary, the roster, its log and the instruments are shipped to `_report`. The machine is given back (`RELEASE`) only when the report and every finished job were shipped; if an upload failed, the machine is kept and the cell names the jobs.

In [ ]:
# C3 - the job list, each finished job shipped, a progress line a minute, the summary, the machine
from geolip.modular_diffusion import jobs
RELEASE = True                     # give the machine back at the end (only when every finished job and the report were shipped)
PROGRESS_S = 60
MISSING = earlier_missing()
assert not MISSING, f"the earlier runs' jobs are not all here ({', '.join(MISSING)}): STOP before the roster, nothing is trained here"
tags = [t for t, *_ in JOBS]
roster = f"{OUT}/roster.txt"
cap = lambda t: max(30, int(3 * COST[t] / 60))
with open(roster, "w", encoding="utf-8", newline="\n") as fh:
    fh.write("\n".join(f"{t} | {cap(t)} | geolip.modular_diffusion.run | {BY_TAG[t][0]}" for t in tags) + "\nEND\n")
fetch_back(tags)
for t in tags:      # a claim left by an interrupted session would hold its job forever
    if mark(t, "CLAIM") and not (mark(t, "DONE") or mark(t, "FAILED")):
        os.remove(f"{RUNS}/{t}/CLAIM")
def after_job(tag, folder, ok):
    if ok and REPO and not ship_job(tag, folder):
        print(f"{tag}: NOT SHIPPED (its folder stays on this machine: {folder})", flush=True)

shipped_or_fetched = lambda t: mark(t, "SHIPPED") or mark(t, "FROM_HUB")
for t in tags:      # a job that finished in an earlier session but whose upload failed: shipped again now
    if REPO and mark(t, "DONE") and not shipped_or_fetched(t):
        after_job(t, f"{RUNS}/{t}", True)

t0 = time.time()
left0 = sum(COST[t] for t in tags if not (mark(t, "DONE") or mark(t, "FAILED")))
print(f"start: {sum(mark(t, 'DONE') for t in tags)}/{len(tags)} jobs already done; about {left0 / 60:.0f} min to run; one runner")
runner = threading.Thread(target=jobs.run_roster, daemon=True,
                          kwargs=dict(roster=roster, out=RUNS, stop=f"{OUT}/STOP", log=f"{OUT}/roster.log", card_room_mib=None,
                                      ram_free_gb=4.0, max_hours=12.0, claim=True, after_job=after_job, wait_s=5.0,
                                      env={"OMP_NUM_THREADS": "4", "MKL_NUM_THREADS": "4", "HF_HUB_OFFLINE": "1"}))
runner.start()
while runner.is_alive():
    time.sleep(PROGRESS_S)
    done, failed = sum(mark(t, "DONE") for t in tags), sum(mark(t, "FAILED") for t in tags)
    left = sum(COST[t] for t in tags if not (mark(t, "DONE") or mark(t, "FAILED")))
    spent = (time.time() - t0) / 60
    rate = (left0 - left) / 60 / spent if left0 > left and spent > 0 else 1.0     # projected minutes done a minute
    print(f"jobs done {done}/{len(tags)} (failed {failed}); {spent:.0f} min spent; about {left / 60 / rate:.0f} min left", flush=True)

# the summary: each job's exchanged reads beside the unexchanged ones, and its exactness read
lines = []
for t in tags:
    try:            # a summary line never stops the report and the machine's return
        p = f"{RUNS}/{t}/result.json"
        if not os.path.exists(p):
            lines.append(f"{t}: NO RESULT" + (" (FAILED)" if mark(t, "FAILED") else ""))
            continue
        r = json.load(open(p, encoding="utf-8"))
        x = r["exact"]
        ex = (f"exact {x['holds']} (largest outside {x['largest_outside']}, before the clamp {x['largest_outside_unclamped']}); "
              f"changed something {x['real']} (exchanged cells that differ {x['inside_share']})")
        if r["variant"] == "compose":
            pooled = lambda b: sum(sum(v["per_list"]["n_right"]) for v in b.values()) / sum(int(n) * len(v["per_list"]["n_right"]) for n, v in b.items())
            lines.append(f"{t}: things right {pooled(r['by_n']):.4f} exchanged, {pooled(r['unswapped']):.4f} unexchanged; "
                         f"star {r['by_n'].get('9', {}).get('by_class', {}).get('8')}; {ex}")
        else:
            a, b = r["tables"], r["tables_unswapped"]
            lines.append(f"{t}: picture right {a['all']['P']['mean']:.3f} / {b['all']['P']['mean']:.3f} over the six, "
                         f"{a['families']['combined']['P']['mean']:.3f} / {b['families']['combined']['P']['mean']:.3f} on combined "
                         f"(exchanged / unexchanged); {ex}")
    except Exception as e:
        lines.append(f"{t}: the summary line failed: {e!r}"[:300])
summary = "\n".join(lines) + "\n"
print(summary)
REPORT = f"{OUT}/report"
os.makedirs(REPORT, exist_ok=True)
open(f"{REPORT}/summary.md", "w", encoding="utf-8").write(summary)
json.dump(INSTRUMENTS, open(f"{REPORT}/instruments.json", "w", encoding="utf-8"), indent=1)
for f in ("roster.txt", "roster.log"):
    if os.path.exists(f"{OUT}/{f}"):
        shutil.copy(f"{OUT}/{f}", f"{REPORT}/{f.replace('.log', '_log.txt')}")
for t in tags:      # the logs of the jobs that failed
    if mark(t, "FAILED"):
        for f in ("run.log", "log.txt"):
            if os.path.exists(f"{RUNS}/{t}/{f}"):
                shutil.copy(f"{RUNS}/{t}/{f}", f"{REPORT}/failed_{t}_{f.replace('.', '_')}.txt")
shipped = hub.ship(REPORT, REPO, f"{PREFIX}/_report", "toy swap: summary, roster and instruments") if REPO else None
unshipped = [t for t in tags if mark(t, "DONE") and not shipped_or_fetched(t)]
unfinished = [t for t in tags if not mark(t, "DONE")]
print("report shipped:", shipped, "| finished jobs not on the Hub:", ", ".join(unshipped) or "none",
      "| jobs not finished:", ", ".join(unfinished) or "none")
if RELEASE and shipped and not unshipped and not unfinished:
    from google.colab import runtime
    runtime.unassign()
else:
    print("THE MACHINE IS KEPT:", "the report was not shipped" if not shipped else
          f"these finished jobs were not shipped: {', '.join(unshipped)} (run C3 again to retry)" if unshipped else
          f"these jobs did not finish: {', '.join(unfinished)}" if unfinished else "RELEASE is False")